# In-hospital mortality with Qwen3 (zero-shot LLM)

Reconstruction of the `llm/` pipeline (`main.py`, `inference.py`, `data_processing.py`, `evaluate.py`,
`tasks/mimic_mortality/configs/qwen*_inf.py`) as a single notebook.

* **Input**: `data/mimic_mortality/llm_test.parquet` from `01_prepare_data.ipynb`. It holds the last 36 h before the prediction time, at 30-minute resolution, as text lines `hours - item: value unit; ...`.
* **Prompt**: the system prompt from `llm/tasks/mimic_mortality/prompts/prompt_template.txt`, with the horizon fixed to 12 hours throughout (the original says "next hour" in two places).
* **Output parsing**: `Prediction: Yes/No, Risk: <0..1>`. The risk score is the probability used for AUROC/AUPRC. When no risk can be parsed, the binary prediction (0/1) is used and the case is counted.
* **Backends**: `vllm` (what the paper used; Linux + CUDA, A100 80GB for Qwen3-32B) or plain `transformers` (`hf`) for smaller models / quick tests.

**PhysioNet policy:** MIMIC data must not be sent to hosted API providers. Run the model locally
(see https://physionet.org/news/post/gpt-responsible-use).

In [ ]:
import sys
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))
from mortality_common import DATA_DIR, RESULTS_DIR, binary_metrics, format_metrics

In [ ]:
# ---------------- configuration (mirrors llm/tasks/mimic_mortality/configs/qwen32b_inf.py) ----------------
MODEL_ID = 'Qwen/Qwen3-32B'      # or 'Qwen/Qwen3-4B'
BACKEND = 'vllm'                 # 'vllm' or 'hf'
TEST_FILE = DATA_DIR / 'llm_test.parquet'
NUM_INF_ROWS = None              # e.g. 200 for a quick run (rows are shuffled with SEED first)
SEED = 1234

MAX_MODEL_LEN = 14000            # prompt + completion tokens
MAX_NEW_TOKENS = 2048
ENABLE_THINKING = False          # Qwen3 thinking mode
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
MIN_P = 0.0
PRESENCE_PENALTY = 1.5
TENSOR_PARALLEL_SIZE = 1

run_name = f'qwen_{MODEL_ID.split("/")[-1].lower()}' + ('_think' if ENABLE_THINKING else '')

In [ ]:
SYSTEM_PROMPT = """You are a medical AI assistant focused on predicting mortality events in the Intensive Care Unit (ICU) setting. Your primary task is to analyze provided Electronic Health Record (EHR) data to assess the likelihood of a patient experiencing mortality (death) within the next 12 hours from the last recorded data point.

The input will be a chronologically ordered sequence of patient data from the ICU stay and monitoring. Each entry follows the format: <time_stamp> - <data>, where <time_stamp> denotes the relative time in hours since ICU admission. The data includes patient demographics, laboratory values, vital signs, physiological parameters (ventilator settings, hemodynamic monitoring), fluid inputs/outputs and procedures.

Pay close attention to trends and critical factors in the following parameters, as they relate to mortality risk in critically ill patients:

    Vital Signs & Physiological Parameters: Deteriorating vital signs (hypotension, significant tachycardia or bradycardia, refractory fever/hypothermia, hypoxemia, increasing respiratory rate), increasing pressor/inotrope requirements, worsening ventilatory status (increasing FiO2, PEEP, or difficulty ventilating), falling GCS or neurological deterioration, poor urine output.
    Laboratory Values: Critically abnormal or rapidly worsening lab values such as severe electrolyte imbalances (potassium, sodium, calcium, magnesium, phosphate), worsening renal function (creatinine, BUN), hepatic dysfunction (bilirubin, INR), severe acidosis or alkalosis (ABG pH, pCO2, bicarbonate, lactate), severe hyperglycemia or hypoglycemia, severe hematologic abnormalities (critically low hemoglobin, platelets, or white blood cell count), elevated inflammatory markers (CRP, procalcitonin).
    Medications & Interventions: Escalation of life-sustaining support (e.g., initiation or dose increase of multiple vasopressors, initiation of mechanical ventilation or significant increases in ventilator support, need for urgent dialysis, large volume blood transfusions, initiation of sedation for aggressive ventilation).
    Patient History & Acute Event: Severity of the underlying reason for ICU admission (e.g., severe sepsis, cardiogenic shock, respiratory failure, major trauma, post-cardiac arrest), presence of multiple organ dysfunction, relevant comorbidities.

To arrive at your prediction, follow these reasoning steps:

1. Analyze Recent Data Points: Evaluate recent vital signs, physiological parameters, and laboratory readings for trends, focusing particularly on the rate and direction of change and deviations from normal or target ranges. Note the timing relative to interventions and any acute events.
2. Identify Risk Factors: Assess specific factors that may contribute to imminent mortality risk in this patient at this time, such as acute organ dysfunction or failure, hemodynamic instability, respiratory compromise, neurological deterioration, evidence of sepsis or shock. Note any escalation of required support.
3. Risk Synthesis & Prediction: Combine all findings, including the temporal trends, the severity of acute issues, the number and progression of organ failures and the response to interventions, to determine the likelihood of the patient experiencing mortality within the next 12 hours.

Think step-by-step, and then provide the answer. When writing your final answer, use the following format: "Prediction: <Yes/No>, Risk: <0...1>". 'Yes' indicates a high risk of mortality within the next 12 hours, 'No' indicates a low risk, and 'Risk' is a numerical value between 0 (no risk) and 1 (high risk). For example, "Prediction: Yes, Risk: 0.92", indicates a very high risk of mortality. Base your analysis SOLELY on the provided EHR text and your embedded medical knowledge."""

USER_TEMPLATE = """Patient Data:
{ehr_text}

Analyze the patient first, and then provide your final prediction about mortality risk in the next 12 hrs in the correct format."""

## Load data and build prompts
Prompts that are too long are shortened by dropping the **oldest** timeline lines. The original truncated the end
of the prompt, which cut off the most recent measurements and the instructions.

In [ ]:
from transformers import AutoTokenizer

test_df = pd.read_parquet(TEST_FILE)
if NUM_INF_ROWS is not None:
    test_df = test_df.sample(n=min(NUM_INF_ROWS, len(test_df)), random_state=SEED).reset_index(drop=True)
print(f'{len(test_df)} stays, {int(test_df["Label"].sum())} deaths')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
PROMPT_BUDGET = MAX_MODEL_LEN - MAX_NEW_TOKENS


def build_prompt(ehr_text):
    return tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': USER_TEMPLATE.format(ehr_text=ehr_text)}],
        tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING,
    )


def fit_prompt(text):
    """Drop the oldest timeline lines until the prompt fits into PROMPT_BUDGET tokens."""
    header, _, timeline = text.partition('Timeline')
    first_line, _, rows = timeline.partition('\n')
    rows = rows.split('\n')
    prompt = build_prompt(text)
    n_tokens = len(tokenizer(prompt)['input_ids'])
    truncated = False
    while n_tokens > PROMPT_BUDGET and len(rows) > 1:
        # remove a block of rows proportional to the overflow (fewer re-tokenisations)
        n_drop = max(1, int(len(rows) * (n_tokens - PROMPT_BUDGET) / n_tokens) + 1)
        rows = rows[n_drop:]
        prompt = build_prompt(header + 'Timeline' + first_line + '\n' + '\n'.join(rows))
        n_tokens = len(tokenizer(prompt)['input_ids'])
        truncated = True
    return prompt, n_tokens, truncated


fitted = [fit_prompt(t) for t in tqdm(test_df['Text'], desc='prompts')]
prompts = [f[0] for f in fitted]
prompt_lengths = np.array([f[1] for f in fitted])
print(f'prompt tokens: median {int(np.median(prompt_lengths))}, max {prompt_lengths.max()} | truncated: {sum(f[2] for f in fitted)}')
print(prompts[0][-2500:])

## Generate

In [ ]:
if BACKEND == 'vllm':
    from vllm import LLM, SamplingParams

    llm = LLM(model=MODEL_ID, tensor_parallel_size=TENSOR_PARALLEL_SIZE, max_model_len=MAX_MODEL_LEN, dtype='auto', seed=SEED)
    sampling_params = SamplingParams(
        temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, min_p=MIN_P,
        presence_penalty=PRESENCE_PENALTY, max_tokens=MAX_NEW_TOKENS, seed=SEED,
    )
    outputs = llm.generate(prompts, sampling_params)
    generations = [o.outputs[0].text for o in outputs]
else:
    import torch
    from transformers import AutoModelForCausalLM

    torch.manual_seed(SEED)
    hf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype='auto', device_map='auto')
    hf_model.eval()
    generations = []
    for prompt in tqdm(prompts, desc='generate'):
        inputs = tokenizer(prompt, return_tensors='pt').to(hf_model.device)
        with torch.no_grad():
            out = hf_model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=TEMPERATURE > 0, temperature=TEMPERATURE,
                top_p=TOP_P, top_k=TOP_K, min_p=MIN_P, repetition_penalty=1.0,
            )
        generations.append(tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## Parse predictions and evaluate

In [ ]:
PRED_RE = re.compile(r'Prediction\W*\s*(Yes|No)\b', re.IGNORECASE)
RISK_RE = re.compile(r'Risk\W*\s*([01](?:\.\d+)?|\.\d+)', re.IGNORECASE)


def parse_generation(text):
    """Use the last 'Prediction:' / 'Risk:' (the final answer comes after the reasoning)."""
    answer = text.split('</think>')[-1]
    preds = PRED_RE.findall(answer)
    risks = RISK_RE.findall(answer)
    prediction = preds[-1].lower() == 'yes' if preds else None
    risk = float(risks[-1]) if risks else None
    return prediction, risk


parsed = [parse_generation(g) for g in generations]
results = test_df[['subject_id', 'stay_id', 'Label']].copy()
results['prediction'] = [p for p, _ in parsed]
results['risk'] = [r for _, r in parsed]
results['generation'] = generations

n_no_pred, n_no_risk = results['prediction'].isna().sum(), results['risk'].isna().sum()
print(f'unparsed prediction: {n_no_pred} | unparsed risk: {n_no_risk} (of {len(results)})')
pred_binary = results['prediction'].fillna(False).astype(float)  # no answer -> 'No'
results['prob'] = results['risk'].fillna(pred_binary).clip(0, 1)

metrics = binary_metrics(results['Label'].astype(int), results['prob'])
# the LLM's own Yes/No decision (instead of thresholding the risk at 0.5)
metrics_yes_no = binary_metrics(results['Label'].astype(int), pred_binary)
metrics.update({'f1_yes_no': metrics_yes_no['f1'], 'precision_yes_no': metrics_yes_no['precision'],
                'recall_yes_no': metrics_yes_no['recall'], 'n_unparsed_prediction': int(n_no_pred), 'n_unparsed_risk': int(n_no_risk)})
print('[test]', format_metrics(metrics))
print(f'Yes/No answers: F1={metrics["f1_yes_no"]:.4f} precision={metrics["precision_yes_no"]:.4f} recall={metrics["recall_yes_no"]:.4f}')

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_parquet(RESULTS_DIR / f'{run_name}_generations.parquet', index=False)
results.rename(columns={'Label': 'label'})[['subject_id', 'label', 'prob']].to_csv(RESULTS_DIR / f'{run_name}_test_predictions.csv', index=False)
config = dict(model=MODEL_ID, backend=BACKEND, num_rows=len(results), max_model_len=MAX_MODEL_LEN, max_new_tokens=MAX_NEW_TOKENS,
              enable_thinking=ENABLE_THINKING, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, presence_penalty=PRESENCE_PENALTY)
with open(RESULTS_DIR / f'{run_name}_metrics.json', 'w') as f:
    json.dump({'test': metrics, 'config': config}, f, indent=2, default=str)
print('saved to', RESULTS_DIR)
results[['subject_id', 'Label', 'prediction', 'risk']].head(10)